# Official ColBERTv2 baseline on BEIR (Colab)

Do pipeline **goc** cua ColBERTv2 voi **dung settings cua ban mini**
(`query_maxlen=32`, `doc_maxlen=180`, `nbits=2`, top-100) de so sanh
`Official` vs `Mini Base` vs `CAD` tren cung mot dataset cho cong bang.

Ket qua ghi ra `MyDrive/colbert-run/<dataset>/official_baseline.json` (file rat nho,
nen chi ton vai KB tren Drive). `Mini_ColBERTv2_MultiBEIR.ipynb` se tu doc file nay.

**Thu tu:** chay notebook nay truoc hoac sau deu duoc (chi can file json ton tai tren Drive).
Can Colab GPU (T4).

> Luu y: so trong paper ColBERTv2 dung `doc_maxlen=300` nen KHONG so truc tiep
> duoc voi ban mini. Notebook nay chay lai official voi `180` cho dong dieu kien.

In [ ]:
# 1. Clone the official ColBERT repo (GPU required for indexing)
!nvidia-smi
!git clone https://github.com/stanford-futuredata/ColBERT.git || true
%cd ColBERT
!pip -q install -e .
!pip -q install faiss-gpu-cu12

In [ ]:
# 2. Pin transformers<5 (colbert-ai compatibility) + helpers
!pip install -q "transformers<5" jedi ninja
import torch, transformers
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("transformers", transformers.__version__)

In [ ]:
# 3. Clone the mini repo (for the baseline scripts) and copy them into ColBERT
%cd /content
!git clone https://github.com/HoangNguyen689/colbert-hoangnd.git || true
!git -C colbert-hoangnd pull
MINI_DIR = "/content/colbert-hoangnd"
!cp {MINI_DIR}/baselines/official/prepare_beir.py /content/ColBERT/
!cp {MINI_DIR}/baselines/official/run_official_colbertv2.py /content/ColBERT/
!cp {MINI_DIR}/baselines/official/evaluate_official.py /content/ColBERT/
!ls /content/ColBERT/*.py

In [ ]:
# 4. Mount Google Drive (shared storage for the tiny official_baseline.json)
import os
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = Path('/content/drive/MyDrive/colbert-run')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print('Drive dir:', DRIVE_DIR)

In [ ]:
# 5. Run official prepare -> index+search -> evaluate for each dataset
#    Settings pinned to match the mini pipeline (query/doc maxlen, nbits=2, k=100).
os.chdir('/content/ColBERT')
DATASETS = ['nfcorpus', 'scidocs', 'fiqa', 'arguana']

for ds in DATASETS:
    print(f'\n==================== {ds} ====================')
    dst = DRIVE_DIR / ds
    dst.mkdir(parents=True, exist_ok=True)
    !python prepare_beir.py --dataset {ds} --out /content/ColBERT/data
    !python run_official_colbertv2.py --dataset {ds} --data /content/ColBERT/data/{ds}_colbert --nbits 2 --k 100 --doc-maxlen 180 --query-maxlen 32
    !python evaluate_official.py --dataset {ds} --json-out {dst}/official_baseline.json

In [ ]:
# 6. Print per-dataset comparison tables (Official vs Mini Base vs IDF/Adaptive/CAD)
#    Official comes from Drive; mini rankings come from Drive if the mini notebook synced them.
import json
from pathlib import Path
import sys
sys.path.insert(0, '/content/colbert-hoangnd')
from evaluation.metrics import evaluate, load_json

VARIANTS = ['official', 'original', 'idf', 'adaptive', 'cad']
LABELS = {'official': 'Official', 'original': 'Mini Base', 'idf': 'IDF',
          'adaptive': 'Adaptive', 'cad': 'CAD'}
DATASETS = ['nfcorpus', 'scidocs', 'fiqa', 'arguana']

print(f"{'Dataset':<14} {'Method':<10} {'nDCG@10':>8} {'MRR@10':>8} {'Recall@100':>11}")
print('-' * 55)
for ds in DATASETS:
    d = DRIVE_DIR / ds
    qrels_path = d / 'qrels_test.json'
    if not qrels_path.exists():
        print(f"{ds:<14} (no mini qrels on Drive - run Mini notebook first)")
        continue
    qrels = load_json(qrels_path)
    metrics = {}
    off = d / 'official_baseline.json'
    if off.exists():
        metrics['official'] = json.loads(off.read_text())
    for name in ['original', 'idf', 'adaptive', 'cad']:
        p = d / f'rankings_{name}.json'
        if p.exists():
            metrics[name] = evaluate(load_json(p), qrels)
    for name in VARIANTS:
        m = metrics.get(name)
        if m:
            print(f"{ds:<14} {LABELS[name]:<10} {m['nDCG@10']:>8.4f} {m['MRR@10']:>8.4f} {m['Recall@100']:>11.4f}")
    print('-' * 55)